# Compare train, validation, and test images

This notebook focuses on the three comparisons we kept from the original version:

- **4. Dimensions, aspect ratio, file size, and format:** check the basic image properties.
- **9. Differences within each class:** locate classes and properties that differ across splits.
- **10. Inspect every class:** compare RGB, brightness, contrast, saturation, and example images for all classes.

The original section numbers are retained for easy reference. Every image is measured; the
dataset is not modified. Run all cells using **Python (ML_dayo)**. The measurements are cached.
All charts are saved as PNG files in the project's **pictures** folder, with filenames beginning
`split_comparison_`. Rerunning replaces these generated files.

### Setup and measurements

These setup cells load the dataset and reuse the measurement cache. If files or measurement
settings have changed, they recompute the measurements. The technical measurement cell is
collapsed by default where supported by the notebook viewer.

Image dimensions come from the original files. Color and texture measurements use an
aspect-preserving thumbnail with a 256-pixel longest edge, without augmentation or model
normalization. `BOTTOM_CROP_PIXELS = 0` measures the whole image; change it to your model's
bottom crop to repeat the comparisons without that annotation region. Example grids always
show the uncropped originals.

In [ ]:
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from IPython.display import display, Markdown
from tqdm.auto import tqdm

# Uses the project directory when launched here, with the supplied absolute path as fallback.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data/carbonate_1223 _merge/PPL-1223').is_dir()),
    Path('/home/d/Documents/Other_projects/Rock_Classification'),
)
DATA_ROOT = PROJECT_ROOT / 'data/carbonate_1223 _merge/PPL-1223'
OUTPUT_DIR = PROJECT_ROOT / 'results/image_split_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PICTURES_DIR = PROJECT_ROOT / 'pictures'
PICTURES_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ['train', 'val', 'test']
PALETTE = {'train': '#2878B5', 'val': '#E58B2A', 'test': '#299D78'}
SEED = 42
ANALYSIS_LONG_EDGE = 256
BOTTOM_CROP_PIXELS = 0
HIST_BINS = 32
COMPUTE_FILE_HASHES = True
FORCE_RECOMPUTE = False
N_WORKERS = min(8, os.cpu_count() or 1)
MIN_CLASS_IMAGES = 5
N_EXAMPLES = 4  # Original-image examples per split, for every class.

assert ANALYSIS_LONG_EDGE >= 16
assert BOTTOM_CROP_PIXELS >= 0
assert HIST_BINS >= 2
assert N_WORKERS >= 1
assert MIN_CLASS_IMAGES >= 2
assert all((DATA_ROOT / split).is_dir() for split in SPLITS), DATA_ROOT
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 90, 'savefig.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False})
pd.set_option('display.max_columns', 20)


saved_figures = []

def finish_figure(fig, filename, h_pad=1.5):
    fig.tight_layout(h_pad=h_pad)
    destination = PICTURES_DIR / filename
    fig.savefig(destination, dpi=150, bbox_inches='tight', facecolor='white')
    saved_figures.append(destination)
    plt.show()
    plt.close(fig)
    return destination

def natural_key(value):
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r'(\d+)', str(value))]

print(f'Dataset: {DATA_ROOT}')
print(f'Cache and CSV reports: {OUTPUT_DIR}')
print(f'Charts: {PICTURES_DIR}')

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}
class_names = sorted({p.name for split in SPLITS for p in (DATA_ROOT / split).iterdir()
                      if p.is_dir() and not p.name.startswith('.')}, key=natural_key)
rows, ignored_files = [], []
for split in SPLITS:
    for path in sorted((DATA_ROOT / split).rglob('*')):
        if not path.is_file():
            continue
        relative = path.relative_to(DATA_ROOT)
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            ignored_files.append(str(relative))
            continue
        if len(relative.parts) < 3:
            raise ValueError(f'Image is outside a class folder: {relative}')
        stat = path.stat()
        rows.append({'split': split, 'class_name': relative.parts[1],
                     'relative_path': relative.as_posix(), 'filename': path.name,
                     'file_size_bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns})
manifest = pd.DataFrame(rows)
if manifest.empty:
    raise ValueError(f'No supported images found under {DATA_ROOT}')
counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=class_names, columns=SPLITS, fill_value=0)

if ignored_files:
    print(f'Ignored {len(ignored_files)} non-image files.')

In [ ]:
MEASUREMENT_VERSION = 1
settings = {'version': MEASUREMENT_VERSION, 'long_edge': ANALYSIS_LONG_EDGE,
            'bottom_crop': BOTTOM_CROP_PIXELS, 'hist_bins': HIST_BINS,
            'file_hashes': COMPUTE_FILE_HASHES,
            'split_order': SPLITS, 'class_order': class_names}
fingerprint = hashlib.sha256(json.dumps(settings, sort_keys=True).encode())
for row in manifest.itertuples(index=False):
    fingerprint.update(json.dumps([row.relative_path, row.file_size_bytes, row.mtime_ns]).encode())
cache_key = fingerprint.hexdigest()
CACHE_TABLE = OUTPUT_DIR / 'image_properties.csv.gz'
CACHE_HIST = OUTPUT_DIR / 'histogram_sums.npz'
CACHE_META = OUTPUT_DIR / 'cache_metadata.json'
CACHE_ERRORS = OUTPUT_DIR / 'unreadable_images.csv'

def measure_image(row):
    record = dict(row)
    path = DATA_ROOT / record['relative_path']
    try:
        raw = path.read_bytes()
        if COMPUTE_FILE_HASHES:
            record['file_sha256'] = hashlib.sha256(raw).hexdigest()
        with Image.open(BytesIO(raw)) as source:
            record.update(width=source.width, height=source.height,
                          mode=source.mode, image_format=source.format,
                          frame_count=getattr(source, 'n_frames', 1))
            record['aspect_ratio'] = source.width / source.height
            record['megapixels'] = source.width * source.height / 1_000_000
            record['file_size_kib'] = len(raw) / 1024
            rgb = ImageOps.exif_transpose(source).convert('RGB')
            if BOTTOM_CROP_PIXELS:
                if BOTTOM_CROP_PIXELS >= rgb.height - 2:
                    raise ValueError('Bottom crop leaves fewer than 3 rows')
                rgb = rgb.crop((0, 0, rgb.width, rgb.height - BOTTOM_CROP_PIXELS))
            # Decode before reducing, so corrupt pixel data is reported.
            rgb.thumbnail((ANALYSIS_LONG_EDGE, ANALYSIS_LONG_EDGE), Image.Resampling.LANCZOS)
            record['analysis_width'], record['analysis_height'] = rgb.size
            pixels = np.asarray(rgb, dtype=np.float32)
            gray = pixels @ np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
            for channel, name in enumerate('rgb'):
                record[f'mean_{name}'] = float(pixels[..., channel].mean())
                record[f'std_{name}'] = float(pixels[..., channel].std())
            record['brightness'] = float(gray.mean())
            record['contrast'] = float(gray.std())
            record['saturation'] = float(np.asarray(rgb.convert('HSV'))[..., 1].mean() / 255)
            record['dark_pct'] = float((gray <= 10).mean() * 100)
            record['bright_pct'] = float((gray >= 245).mean() * 100)
            histogram = np.stack([np.histogram(values, bins=HIST_BINS, range=(0, 256))[0]
                                  for values in [pixels[..., 0], pixels[..., 1], pixels[..., 2], gray]])
            histogram = histogram.astype(np.float64) / gray.size
            probabilities = histogram[3][histogram[3] > 0]
            record['entropy'] = float(-(probabilities * np.log2(probabilities)).sum())
            if min(gray.shape) >= 3:
                laplacian = (gray[:-2, 1:-1] + gray[2:, 1:-1]
                             + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1])
                record['laplacian_var'] = float(laplacian.var())
            else:
                record['laplacian_var'] = np.nan
        return record, histogram, None
    except Exception as exc:
        return None, None, {'relative_path': record['relative_path'], 'split': record['split'],
                            'class_name': record['class_name'], 'error': f'{type(exc).__name__}: {exc}'}

cache_valid = False
if not FORCE_RECOMPUTE and all(p.exists() for p in [CACHE_TABLE, CACHE_HIST, CACHE_META, CACHE_ERRORS]):
    try:
        cache_valid = json.loads(CACHE_META.read_text()).get('cache_key') == cache_key
        if cache_valid:
            with np.load(CACHE_HIST, allow_pickle=False) as saved:
                cache_valid = saved['histogram_sums'].shape == (len(SPLITS), len(class_names), 4, HIST_BINS)
    except (ValueError, OSError, KeyError):
        cache_valid = False
if cache_valid:
    properties = pd.read_csv(CACHE_TABLE)
    errors = pd.read_csv(CACHE_ERRORS)
    with np.load(CACHE_HIST, allow_pickle=False) as saved:
        histogram_sums = saved['histogram_sums'].copy()
    print(f'Loaded cached measurements for {len(properties):,} images.')
else:
    histogram_sums = np.zeros((len(SPLITS), len(class_names), 4, HIST_BINS), dtype=np.float64)
    records, failed = [], []
    split_index = {value: i for i, value in enumerate(SPLITS)}
    class_index = {value: i for i, value in enumerate(class_names)}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        measured = pool.map(measure_image, manifest.to_dict('records'))
        for record, histogram, error in tqdm(measured, total=len(manifest), desc='Measuring images'):
            if error is not None:
                failed.append(error)
                continue
            records.append(record)
            histogram_sums[split_index[record['split']], class_index[record['class_name']]] += histogram
    properties = pd.DataFrame(records)
    errors = pd.DataFrame(failed, columns=['relative_path', 'split', 'class_name', 'error'])
    if properties.empty:
        display(errors.head(10))
        raise RuntimeError('No images could be decoded; inspect the paths and errors above.')
    properties.to_csv(CACHE_TABLE, index=False, compression='gzip')
    errors.to_csv(CACHE_ERRORS, index=False)
    np.savez_compressed(CACHE_HIST, histogram_sums=histogram_sums)
    CACHE_META.write_text(json.dumps({'cache_key': cache_key, 'settings': settings,
                                      'images_measured': len(properties)}, indent=2))

valid_counts = pd.crosstab(properties['class_name'], properties['split']).reindex(
    index=class_names, columns=SPLITS, fill_value=0)
coverage = pd.DataFrame({'discovered': counts.sum(), 'measured': valid_counts.sum()})
coverage['failed'] = coverage['discovered'] - coverage['measured']
display(coverage)
if not errors.empty:
    display(errors.head(20))
    print('The plots below use successfully decoded images only. See unreadable_images.csv.')
if (coverage['measured'] == 0).any():
    raise RuntimeError('At least one split has no decodable images.')
assert len(properties) + len(errors) == len(manifest)
assert np.allclose(histogram_sums.sum(axis=-1), valid_counts.T.to_numpy()[:, :, None])

## 4. Dimensions, aspect ratio, file size, and format

**Question:** do the splits have the same image dimensions and similar file sizes?

The y-axis is the **fraction of images at or below the x-axis value**. For example,
0.70 at 600 KiB means 70% of images are 600 KiB or smaller. At y = 0.50 you can read the
median. A curve farther right means larger values; overlapping curves mean similar distributions.

When every image has the same width, height, or aspect ratio, the curves overlap as a vertical
line. Larger files do not automatically mean better quality: compression and image detail
also affect file size.

In [ ]:
dimension_table = properties.groupby(['split', 'width', 'height']).size().rename('images').reset_index()
display(dimension_table)
display(pd.crosstab(properties['split'], [properties['mode'], properties['image_format']]).reindex(SPLITS))
display(properties.groupby('split')[['analysis_width', 'analysis_height', 'frame_count']].agg(['min', 'max']).reindex(SPLITS))

def plot_ecdf(ax, data, metric, title, xlabel):
    for split in SPLITS:
        values = np.sort(data.loc[data['split'].eq(split), metric].dropna().to_numpy())
        if not len(values):
            continue
        # Include a zero-height step, including for constant-valued distributions.
        x = np.r_[np.nextafter(values[0], -np.inf), values]
        y = np.r_[0.0, np.arange(1, len(values) + 1) / len(values)]
        ax.step(x, y, where='post', color=PALETTE[split], alpha=0.85,
                linestyle={'train': '-', 'val': '--', 'test': ':'}[split], label=split)
    ax.set(title=title, xlabel=xlabel, ylabel='Fraction of images ≤ x', ylim=(0, 1.02))
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric, title, units in zip(axes.flat,
        ['width', 'height', 'aspect_ratio', 'file_size_kib'],
        ['Native width', 'Native height', 'Native aspect ratio', 'File size'],
        ['Pixels', 'Pixels', 'Width / height', 'KiB']):
    plot_ecdf(ax, properties, metric, title, units)
finish_figure(fig, 'split_comparison_dimensions_and_file_size.png')
print('Median file size (KiB):')
display(properties.groupby('split')['file_size_kib'].median().reindex(SPLITS).round(1))

## 9. Do the differences persist within the same class?

**Question:** which classes look different across splits, and in which properties?

Each row is one class. The left panel compares validation with training; the right compares
test with training. **Red means higher than training; blue means lower.** A nearly white cell
means the averages are close relative to the variation among images. It does not prove that
the full distributions are identical.

The color represents the difference in means divided by the combined spread of the two groups
(the standardized mean difference). Stronger color means a larger difference on that scale.
Colors stop getting stronger beyond ±2; full values are saved in the CSV.

**Gray cells are omitted:** either group has fewer than five images, or both groups have zero
variation. Check the count table below: even five images is a small sample.

| Property | Higher values mean… |
| :--- | :--- |
| Mean red / green / blue | More of that channel on average, on a 0–255 scale. |
| Brightness | Lighter images on average. |
| Contrast | More variation between light and dark pixels within images. |
| Saturation | Stronger color, rather than gray tones. |
| Intensity variety | A more varied mix of pixel intensities; measured by histogram entropy. |
| Local detail | Stronger small-scale intensity changes; affected by both rock texture and focus. |

Use this overview to find a class, then inspect its charts and examples in section 10.

In [ ]:
properties['log_laplacian_var'] = np.log1p(properties['laplacian_var'])
METRIC_LABELS = {
    'mean_r': 'Mean red', 'mean_g': 'Mean green', 'mean_b': 'Mean blue',
    'brightness': 'Brightness', 'contrast': 'Contrast', 'saturation': 'Saturation',
    'entropy': 'Intensity variety', 'log_laplacian_var': 'Local detail (log scale)',
}

def distribution_distances(reference, comparison):
    a = np.sort(np.asarray(reference, dtype=float))
    b = np.sort(np.asarray(comparison, dtype=float))
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if not len(a) or not len(b):
        return np.nan, np.nan
    grid = np.unique(np.r_[a, b])
    ks = np.max(np.abs(np.searchsorted(a, grid, side='right') / len(a)
                       - np.searchsorted(b, grid, side='right') / len(b)))
    pooled_sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if min(len(a), len(b)) > 1 else np.nan
    smd = (b.mean() - a.mean()) / pooled_sd if pooled_sd > 0 else np.nan
    return float(ks), float(smd)

class_comparison_rows = []
WITHIN_CLASS_METRICS = ['mean_r', 'mean_g', 'mean_b', 'brightness', 'contrast',
                        'saturation', 'entropy', 'log_laplacian_var']
for class_name in class_names:
    subset = properties.loc[properties['class_name'].eq(class_name)]
    for comparison in ['val', 'test']:
        for metric in WITHIN_CLASS_METRICS:
            a = subset.loc[subset['split'].eq('train'), metric].dropna()
            b = subset.loc[subset['split'].eq(comparison), metric].dropna()
            enough = min(len(a), len(b)) >= MIN_CLASS_IMAGES
            ks, smd = distribution_distances(a, b) if enough else (np.nan, np.nan)
            class_comparison_rows.append({'class_name': class_name, 'comparison': comparison,
                                           'metric': metric, 'n_train': len(a), 'n_comparison': len(b),
                                           'train_mean': a.mean(), 'comparison_mean': b.mean(),
                                           'enough_images': enough, 'smd': smd, 'ks_distance': ks})
class_comparisons = pd.DataFrame(class_comparison_rows)
fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharey=True)
for ax, split in zip(axes, ['val', 'test']):
    grid = class_comparisons.loc[class_comparisons['comparison'].eq(split)].pivot(
        index='class_name', columns='metric', values='smd').reindex(index=class_names, columns=WITHIN_CLASS_METRICS)
    grid.columns = [METRIC_LABELS[m] for m in grid.columns]
    ax.set_facecolor('#eeeeee')
    sns.heatmap(grid, cmap='RdBu_r', center=0, vmin=-2, vmax=2, ax=ax,
                linewidths=0.4, cbar_kws={'label': 'Standardized mean difference'})
    ax.set(title=f'{split} minus train, within class', xlabel='', ylabel='')
    ax.tick_params(axis='x', rotation=45)
finish_figure(fig, 'split_comparison_within_class_heatmap.png')
display(valid_counts.style.background_gradient(cmap='Blues', axis=None))
class_comparisons.to_csv(OUTPUT_DIR / 'within_class_comparisons.csv', index=False)
class_comparisons.to_csv(PICTURES_DIR / 'split_comparison_within_class_statistics.csv', index=False)

## 10. Inspect every class

**Question:** for this rock class, how do train, validation, and test compare?

Every class below has two saved figures:

1. **Property charts:** mean red, green, blue, brightness, contrast, and saturation.
2. **Example images:** up to four original images per split, chosen with a fixed random seed.

Each box summarizes measurements from **all readable images of that class and split**.
The line inside is the median; the box contains the middle 50%. A higher box means higher
values, and a taller box means more variation in the middle half of the images. Whiskers extend
to observed values within 1.5 box heights of the box edges. Extreme points are hidden for large
groups but still included in the measurements. Groups with fewer than 20 images also show
individual dots. `n` is the number of images; a single image gives only a dot/line, not a useful distribution.

**Axes use the same limits across all classes**, making comparisons between class charts easier.
RGB and brightness use 0–255; saturation uses 0–1. Contrast is the within-image standard deviation
of brightness. Different properties have different units, so do not compare their box heights
directly. Example images are a small visual sample, not the complete distribution.

Look for direction and overlap, not just whether the medians are exactly equal. A difference
does not identify its cause: lighting, sample content, annotations, or labels may contribute.
The small filename-prefix tables are clues for inspection, not proof of correct rock labels.

In [ ]:
CLASS_METRICS = ['mean_r', 'mean_g', 'mean_b', 'brightness', 'contrast', 'saturation']
METRIC_UNITS = {
    'mean_r': 'Mean red intensity (0–255)',
    'mean_g': 'Mean green intensity (0–255)',
    'mean_b': 'Mean blue intensity (0–255)',
    'brightness': 'Mean brightness (0–255)',
    'contrast': 'Within-image brightness standard deviation',
    'saturation': 'Mean saturation (0–1)',
}
METRIC_LIMITS = {metric: (0, 255) for metric in CLASS_METRICS}
METRIC_LIMITS.update(contrast=(0, 128), saturation=(0, 1))
np.random.seed(SEED)  # Reproducible jitter for dots in small groups.
export_rows = []
example_rows = []

def inspect_class(class_name):
    selected = properties.loc[properties['class_name'].eq(class_name)].copy()
    split_counts = selected.groupby('split').size().reindex(SPLITS, fill_value=0)
    class_slug = re.sub(r'[^A-Za-z0-9_-]', '_', class_name)
    if re.fullmatch(r'class\d+', class_name):
        class_slug = f'class{int(class_name[5:]):02d}'
    display(Markdown(f'### {class_name}'))
    display(Markdown(' | '.join(f'**{split}: {split_counts[split]:,} images**' for split in SPLITS)))
    if (split_counts < MIN_CLASS_IMAGES).any():
        display(Markdown('**Small sample:** at least one split has fewer than five images. '
                         'Treat its box and example images as descriptions of those few images.'))
    selected['filename_prefix'] = selected['filename'].str.extract(r'^([^.0-9_]+)', expand=False).fillna('(other)')
    display(pd.crosstab(selected['filename_prefix'], selected['split']).reindex(columns=SPLITS, fill_value=0))

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, metric in zip(axes.flat, CLASS_METRICS):
        sns.boxplot(data=selected, x='split', y=metric, order=SPLITS,
                    hue='split', hue_order=SPLITS, palette=PALETTE, legend=False,
                    showfliers=False, ax=ax)
        small_groups = selected.loc[selected['split'].isin(split_counts[split_counts < 20].index)]
        if not small_groups.empty:
            sns.stripplot(data=small_groups, x='split', y=metric, order=SPLITS,
                          color='#252525', jitter=0.12, size=4, alpha=0.7, ax=ax)
        ax.set(title=METRIC_LABELS[metric], xlabel='', ylabel=METRIC_UNITS[metric],
               ylim=METRIC_LIMITS[metric])
        ax.set_xticks(range(len(SPLITS)), [f'{split}\n(n={split_counts[split]:,})' for split in SPLITS])
    fig.suptitle(f'{class_name}: image properties by split', fontsize=16)
    property_path = finish_figure(fig, f'split_comparison_{class_slug}_properties.png', h_pad=2.5)

    fig, axes = plt.subplots(len(SPLITS), N_EXAMPLES, figsize=(14, 10.5), squeeze=False)
    for row, split in enumerate(SPLITS):
        subset = selected.loc[selected['split'].eq(split)]
        examples = subset.sample(n=min(N_EXAMPLES, len(subset)), random_state=SEED)
        for col in range(N_EXAMPLES):
            ax = axes[row, col]
            ax.axis('off')
            if col >= len(examples):
                ax.set_title(f'{split}: no additional image', fontsize=9)
                continue
            example = examples.iloc[col]
            with Image.open(DATA_ROOT / example['relative_path']) as source:
                preview = ImageOps.exif_transpose(source).convert('RGB')
                preview.thumbnail((400, 300), Image.Resampling.LANCZOS)
                ax.imshow(preview)
            ax.set_title(f"{split} · {class_name}\n{example['filename']}", fontsize=9)
            example_rows.append({'class_name': class_name, 'split': split, 'position': col + 1,
                                 'relative_path': example['relative_path']})
    fig.suptitle(f'{class_name}: original images from each split', fontsize=16)
    example_path = finish_figure(fig, f'split_comparison_{class_slug}_examples.png', h_pad=3.0)
    export_rows.append({'class_name': class_name, 'properties_chart': property_path.name,
                        'examples_chart': example_path.name,
                        **{f'n_{split}': int(split_counts[split]) for split in SPLITS}})

for class_name in class_names:
    inspect_class(class_name)

class_statistics = properties.groupby(['class_name', 'split'])[CLASS_METRICS].agg(
    ['count', 'mean', 'std', 'median', 'min', 'max'])
class_statistics.to_csv(PICTURES_DIR / 'split_comparison_class_statistics.csv')
pd.DataFrame(export_rows).to_csv(PICTURES_DIR / 'split_comparison_chart_index.csv', index=False)
pd.DataFrame(example_rows).to_csv(PICTURES_DIR / 'split_comparison_example_images.csv', index=False)
assert len(export_rows) == len(class_names)
assert all(path.is_file() for path in saved_figures)
print(f'Saved {len(saved_figures)} PNG figures for {len(class_names)} classes plus the two overview charts.')
print(f'Output folder: {PICTURES_DIR}')
print('CSV files contain class statistics, within-class comparisons, the chart index, and example filenames.')